In [0]:
%run ./config_init

In [0]:
df = spark.table(f"{catalog_name}.{schema_config}.configurations_table")
config_table_schema = df.schema

In [0]:
bronze_config_values = [
    ["landing", "bronze", "files_landing", "move_files", "volume", "/Volumes/medisure/bronze/in/", "", "/Volumes/medisure/bronze/todo/", ""],
    ["archive", "bronze", "archiving", "move_files", "volume", "/Volumes/medisure/bronze/todo/", "", "/Volumes/medisure/bronze/archive/", ""],
    ["claims_batch", "bronze", "ingest", "append", "csv", "/Volumes/medisure/bronze/todo/claims_batch.csv", "", "medisure.bronze.bt_fact_claims_batch", ""],
    ["claims_stream", "bronze", "ingest", "append", "json", "/Volumes/medisure/bronze/todo/claims_stream.json", "", "medisure.bronze.bt_fact_claims_stream", ""],
    ["diagnosis", "bronze", "ingest", "overwrite", "csv", "/Volumes/medisure/bronze/todo/diagnosis_ref.csv", "", "medisure.bronze.bt_ref_diagnosis", ""],
    ["members", "bronze", "ingest", "overwrite", "csv", "/Volumes/medisure/bronze/todo/members.csv", "", "medisure.bronze.bt_fact_members", ""],
    ["providers", "bronze", "ingest", "overwrite", "json", "/Volumes/medisure/bronze/todo/claims_batch.csv", "", "medisure.bronze.bt_fact_providers", ""]
]

df = spark.createDataFrame(bronze_config_values, config_table_schema)
df.display()

In [0]:
silver_config_values = [
    ["claims_batch", "silver", "clean", "replace", "delta", "medisure.bronze.bt_fact_claims_batch", "ClaimID,MemberID,ProviderID", "medisure.silver.st_fact_claims_batch", "ClaimID,MemberID,ProviderID"],
    ["claims_stream", "silver", "clean", "replace", "delta", "medisure.bronze.bt_fact_claims_stream", "ClaimID,MemberID,ProviderID", "medisure.silver.st_fact_claims_stream", "ClaimID,MemberID,ProviderID"],
    ["diagnosis", "silver", "clean", "overwrite", "delta", "medisure.bronze.bt_ref_diagnosis", "", "medisure.silver.st_ref_diagnosis", ""],
    ["members", "silver", "clean", "overwrite", "delta", "medisure.bronze.bt_fact_members", "", "medisure.silver.st_fact_members", ""],
    ["providers", "silver", "clean", "overwrite", "delta", "medisure.bronze.bt_fact_providers", "", "medisure.silver.st_fact_providers", ""]
]

df = spark.createDataFrame(silver_config_values, config_table_schema)
df.display()

In [0]:
silver_transformed_config_values = [
    ["claims_batch", "transform", "clean", "replace", "delta", "medisure.silver.st_fact_claims_batch", "ClaimID,MemberID,ProviderID", "medisure.silver.st_fact_claims_batch_transformed", "ClaimID,MemberID,ProviderID"],
    ["claims_stream", "transform", "clean", "replace", "delta", "medisure.silver.st_fact_claims_stream", "ClaimID,MemberID,ProviderID", "medisure.silver.st_fact_claims_stream_transformed", "ClaimID,MemberID,ProviderID"],
    #["diagnosis", "transform", "clean", "overwrite", "delta", "medisure.silver.st_ref_diagnosis", "", "medisure.silver.st_ref_diagnosis_transformed", ""],
    #["members", "transform", "clean", "overwrite", "delta", "medisure.silver.st_fact_members", "", "medisure.silver.st_fact_members_transformed", ""],
    #["providers", "transform", "clean", "overwrite", "delta", "medisure.silver.st_fact_providers", "", "medisure.silver.st_fact_providers_transformed", ""]
]

df = spark.createDataFrame(silver_transformed_config_values, config_table_schema)
df.display()

In [0]:
gold_config_values = [
    ["claims_batch", "gold", "curate", "replace", "delta", "medisure.silver.st_fact_claims_batch_transformed", "ClaimID,MemberID,ProviderID", "medisure.gold.gt_fact_claims_batch", "ClaimID,MemberID,ProviderID"],
    ["claims_stream", "gold", "curate", "replace", "delta", "medisure.silver.st_fact_claims_stream", "ClaimID,MemberID,ProviderID", "medisure.gold.gt_fact_claims_stream", "ClaimID,MemberID,ProviderID"],
    ["diagnosis", "gold", "curate", "overwrite", "delta", "medisure.silver.st_ref_diagnosis", "", "medisure.gold.gt_ref_diagnosis", ""],
    ["members", "gold", "curate", "overwrite", "delta", "medisure.silver.st_fact_members", "", "medisure.gold.gt_fact_members", ""],
    ["providers", "gold", "curate", "overwrite", "delta", "medisure.silver.st_fact_providers", "", "medisure.gold.gt_fact_providers", ""]
]

df = spark.createDataFrame(gold_config_values, config_table_schema)
df.display()

In [0]:
combine_configs = bronze_config_values + silver_config_values + silver_transformed_config_values + gold_config_values
combined_df = spark.createDataFrame(combine_configs, config_table_schema)

df = write_table(combined_df, f"{catalog_name}.{schema_config}.configurations_table", "overwrite")
df.orderBy("medallion_layer").display()